# SRP182008.h5ad 数据结构分析

本 notebook 用于分析单细胞 RNA-seq 数据集的结构，帮助理解：
1. **数据结构组成**：AnnData 对象的各个组成部分
2. **scMAE 算法数据使用情况**：哪些数据被使用，哪些被忽略
3. **深度学习模型输入要求**：建立自己模型时需要注意什么

In [1]:
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import issparse
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

DATA_PATH = '../data/SRP182008.h5ad'
print('Libraries imported successfully!')

Libraries imported successfully!


/data/luolie/conda/envs/scclubench-main/lib/python3.9/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


---

## 1. 数据加载与基本信息

In [2]:
# 加载数据
adata = ad.read_h5ad(DATA_PATH)

# 基本信息
print('=' * 60)
print('📊 SRP182008.h5ad 基本信息')
print('=' * 60)
print(f'细胞数 (n_obs): {adata.n_obs:,}')
print(f'基因数 (n_vars): {adata.n_vars:,}')
print(f'矩阵形状 X: {adata.X.shape}')
print(f'数据类型: {type(adata.X)}')

# 数据类型
if issparse(adata.X):
    print(f'稀疏矩阵格式: {type(adata.X).__name__}')
    print(f'非零元素数: {adata.X.nnz:,}')
    print(f'稀疏度: {1 - adata.X.nnz / (adata.X.shape[0] * adata.X.shape[1]):.4f}')
else:
    print(f'密集矩阵 dtype: {adata.X.dtype}')

📊 SRP182008.h5ad 基本信息
细胞数 (n_obs): 13,514
基因数 (n_vars): 53,678
矩阵形状 X: (13514, 53678)
数据类型: <class 'scipy.sparse._csr.csr_matrix'>
稀疏矩阵格式: csr_matrix
非零元素数: 17,378,932
稀疏度: 0.9760


---

## 2. AnnData 数据结构详解

AnnData 对象包含以下核心组件：

| 组件 | 含义 | 维度 |
|------|------|------|
| **X** | 主表达矩阵 | cells × genes |
| **obs** | 细胞元数据 | cells × attributes |
| **var** | 基因元数据 | genes × attributes |
| **layers** | 表达矩阵的不同版本 | cells × genes |
| **obsm** | 细胞的多维嵌入 | cells × k |
| **varm** | 基因的多维嵌入 | genes × k |
| **obsp** | 细胞间距离/邻接 | cells × cells |
| **uns** | 非结构化数据 | dict |
| **raw** | 原始数据备份 | AnnData |

In [3]:
print('=' * 60)
print('📁 AnnData 各组件详细信息')
print('=' * 60)

# X 矩阵
print('\n📌 X (主表达矩阵)')
print(f'   形状: {adata.X.shape}')
print(f'   类型: {type(adata.X).__name__}')

# obs - 细胞元数据
print('\n📌 obs (细胞元数据)')
print(f'   形状: {adata.obs.shape}')
print(f'   列名: {list(adata.obs.columns)}')
print('\n   前5行预览:')
display(adata.obs.head())

📁 AnnData 各组件详细信息

📌 X (主表达矩阵)
   形状: (13514, 53678)
   类型: csr_matrix

📌 obs (细胞元数据)
   形状: (13514, 13)
   列名: ['Orig.ident', 'nCount_RNA', 'nFeature_RNA', 'Percent.mt', 'Seurat_clusters', 'Celltype', 'Dataset', 'Tissue', 'Organ', 'Condition', 'Genotype', 'Libraries', 'ACE']

   前5行预览:


,Orig.ident,nCount_RNA,nFeature_RNA,Percent.mt,Seurat_clusters,Celltype,Dataset,Tissue,Organ,Condition,Genotype,Libraries,ACE
SRX5290443@@_AAACCTGAGAATTCCC-1,SRX5290443,3737.0,2149,0.0,2,Root stele,SRP182008,Root tip,Root,Normal,Col-0,10x Genomics,10 days old seedling
SRX5290443@@_AAACCTGAGACTTTCG-1,SRX5290443,964.0,747,0.0,14,Root hair,SRP182008,Root tip,Root,Normal,Col-0,10x Genomics,10 days old seedling
SRX5290443@@_AAACCTGAGCTGTTCA-1,SRX5290443,1308.0,881,0.0,14,Root hair,SRP182008,Root tip,Root,Normal,Col-0,10x Genomics,10 days old seedling
SRX5290443@@_AAACCTGAGGTAGCCA-1,SRX5290443,1131.0,790,0.0,5,Root hair,SRP182008,Root tip,Root,Normal,Col-0,10x Genomics,10 days old seedling
SRX5290443@@_AAACCTGAGTAATCCC-1,SRX5290443,2451.0,1290,0.0,5,Root hair,SRP182008,Root tip,Root,Normal,Col-0,10x Genomics,10 days old seedling


In [4]:
# var - 基因元数据
print('\n📌 var (基因元数据)')
print(f'   形状: {adata.var.shape}')
print(f'   列名: {list(adata.var.columns)}')
print('\n   前10行预览:')
display(adata.var.head(10))


📌 var (基因元数据)
   形状: (53678, 1)
   列名: ['features']

   前10行预览:


,features
AT1G01010,AT1G01010
AT1G01020,AT1G01020
AT1G01030,AT1G01030
AT1G01040,AT1G01040
AT1G01046,AT1G01046
AT1G01050,AT1G01050
AT1G01060,AT1G01060
AT1G01070,AT1G01070
AT1G01080,AT1G01080
AT1G01090,AT1G01090


In [5]:
# layers - 表达矩阵的不同版本
print('\n📌 layers (表达矩阵变体)')
print(f'   可用层: {list(adata.layers.keys()) if adata.layers else "无"}')
if adata.layers:
    for layer_name, layer_data in adata.layers.items():
        print(f'   - {layer_name}: 形状={layer_data.shape}, 类型={type(layer_data).__name__}')


📌 layers (表达矩阵变体)
   可用层: 无


In [6]:
# obsm - 细胞嵌入/降维结果
print('\n📌 obsm (细胞多维嵌入)')
print(f'   可用嵌入: {list(adata.obsm.keys()) if adata.obsm else "无"}')
if adata.obsm:
    for key in adata.obsm.keys():
        print(f'   - {key}: 形状={adata.obsm[key].shape}')


📌 obsm (细胞多维嵌入)
   可用嵌入: ['X_tsne', 'X_umap']
   - X_tsne: 形状=(13514, 3)
   - X_umap: 形状=(13514, 2)


In [7]:
# varm - 基因嵌入
print('\n📌 varm (基因多维嵌入)')
print(f'   可用嵌入: {list(adata.varm.keys()) if adata.varm else "无"}')
if adata.varm:
    for key in adata.varm.keys():
        print(f'   - {key}: 形状={adata.varm[key].shape}')


📌 varm (基因多维嵌入)
   可用嵌入: 无


In [8]:
# obsp - 细胞间关系矩阵
print('\n📌 obsp (细胞间关系矩阵)')
print(f'   可用矩阵: {list(adata.obsp.keys()) if adata.obsp else "无"}')


📌 obsp (细胞间关系矩阵)
   可用矩阵: 无


In [9]:
# uns - 非结构化数据
print('\n📌 uns (非结构化数据)')
print(f'   键: {list(adata.uns.keys()) if adata.uns else "无"}')


📌 uns (非结构化数据)
   键: 无


In [10]:
# raw - 原始数据备份
print('\n📌 raw (原始数据备份)')
if adata.raw is not None:
    print(f'   是否存在: 是')
    print(f'   X 形状: {adata.raw.X.shape}')
    print(f'   var 形状: {adata.raw.var.shape}')
else:
    print('   是否存在: 否 (数据集未保存原始数据)')


📌 raw (原始数据备份)
   是否存在: 是
   X 形状: (13514, 53678)
   var 形状: (53678, 1)


---

## 3. scMAE 算法数据使用分析

### 3.1 算法数据流程图

```
原始 h5ad 文件
      │
      ▼
┌─────────────────────────────────────────────┐
│  preprocess.py → prepare_data_for_model()  │
│                                             │
│  1. normalize_sc():                        │
│     - Per-cell 归一化                      │
│     - Log1p 变换                          │
│     - HVG 筛选 (默认 top 1000)             │
│     - Z-score 标准化                       │
│                                             │
│  2. 提取:                                 │
│     - X: 预处理后的表达矩阵                │
│     - Y: 细胞类型标签                      │
│     - sf: size factors                     │
└─────────────────────────────────────────────┘
      │
      ▼
┌─────────────────────────────────────────────┐
│  run.py → scMAE 模型                       │
│                                             │
│  使用的数据:                               │
│  ✅ X: 预处理后的基因表达 (cells × 1000)  │
│  ✅ Y: 细胞类型标签 (用于评估/聚类)        │
│                                             │
│  不使用但可用的数据:                       │
│  ⚠️ sf: size factors (未在模型中使用)     │
│  ⚠️ raw: 原始数据 (未使用)                 │
│  ⚠️ obs/varm: 元数据 (未使用)              │
│                                             │
│  完全不使用的组件:                         │
│  ❌ obsm: 细胞嵌入 (未使用)                 │
│  ❌ varm: 基因嵌入 (未使用)                 │
│  ❌ obsp: 细胞间矩阵 (未使用)               │
│  ❌ uns: 非结构化数据 (未使用)              │
└─────────────────────────────────────────────┘
```

In [11]:
# 模拟 scMAE 的数据预处理流程

import sys
import os
sys.path.insert(0, '/home/luolie/biopipeline/dimension-reduction/plantnet')
from methods.DeepLearning.scMAE.preprocess import prepare_data_for_model

print('=' * 60)
print('🔬 模拟 scMAE 数据预处理流程')
print('=' * 60)

X, Y, sf, adata_processed = prepare_data_for_model(
    DATA_PATH,
    size_factors=True,
    filter_min_counts=True,
    logtrans_input=True,
    normalize_input=True
)

print('\n✅ 预处理后的数据:')
print(f'   X 形状: {X.shape} (细胞数 × 基因数)')
print(f'   Y 形状: {Y.shape}')
print(f'   Y 唯一值数: {Y.nunique()}')

ModuleNotFoundError: No module named 'utils'

In [ ]:
# 细胞类型分布
print('\n📊 细胞类型分布:')
cell_type_counts = Y.value_counts()
display(cell_type_counts)

# 可视化
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Set3(np.linspace(0, 1, len(cell_type_counts)))
bars = ax.barh(cell_type_counts.index, cell_type_counts.values, color=colors)
ax.set_xlabel('Number of Cells')
ax.set_ylabel('Cell Type')
ax.set_title('SRP182008: Cell Type Distribution')

# 添加数值标签
for bar, count in zip(bars, cell_type_counts.values):
    ax.text(count + 10, bar.get_y() + bar.get_height()/2, 
            f'{count}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# 数据统计摘要
print('\n📈 预处理后 X 矩阵统计:')
print(f'   数据类型: {X.values.dtype}')
print(f'   最小值: {X.values.min():.4f}')
print(f'   最大值: {X.values.max():.4f}')
print(f'   均值: {X.values.mean():.4f}')
print(f'   标准差: {X.values.std():.4f}')
print(f'   \u4e2d位数: {np.median(X.values):.4f}')

---

## 4. 数据使用情况详细分类

### 4.1 数据分类表格

In [ ]:
# 创建数据使用情况表格
import pandas as pd

data_usage = pd.DataFrame({
    '组件': ['X', 'Y (cell_type)', 'size_factors', 'raw', 'obs', 'var', 
             'layers', 'obsm', 'varm', 'obsp', 'uns'],
    '数据说明': [
        '主表达矩阵 (预处理后)',
        '细胞类型标签',
        '细胞归一化因子',
        '原始未处理数据',
        '细胞元数据 (QC指标等)',
        '基因元数据',
        '表达矩阵变体',
        '细胞嵌入 (PCA/UMAP等)',
        '基因嵌入',
        '细胞间关系矩阵',
        '非结构化配置信息'
    ],
    'scMAE使用': ['✅ 必需', '✅ 必需', '⚠️ 可选', '❌ 不使用', 
                 '❌ 不使用', '❌ 不使用', '❌ 不使用', '❌ 不使用', 
                 '❌ 不使用', '❌ 不使用', '❌ 不使用'],
    '用途': [
        '模型输入 (cells × 1000 HVG)',
        '聚类评估/标签编码',
        '可传入但模型未用',
        '备份原始数据',
        '存储n_counts等QC指标',
        '存储基因名称/ID',
        'norm_log层可用于ZINB等模型',
        '预计算的降维结果',
        '预计算的基因嵌入',
        'KNN图等预计算结果',
        '配置信息'
    ]
})

pd.set_option('display.max_colwidth', None)
display(data_usage)

### 4.2 可视化数据分类

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    数据使用分类（按重要性排序）                             │
├─────────────────────────────────────────────────────────────────────────┤
│  ████████████████  X (必需)     ████████  Y (必需)                       │
│  细胞×基因矩阵       细胞类型标签                                          │
│                                                                         │
│  ████  size_factors (可选)                                               │
│  归一化因子                                                              │
│                                                                         │
│  ░░░░  layers (备选)       ░░░░  obs/var (元数据)                        │
│  norm_log层              QC信息                                          │
│                                                                         │
│  ▓▓▓▓  obsm/varm/obsp/uns (未使用)                                       │
│  预计算结果/配置                                                          │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# 可视化数据使用分类
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 图1: 数据大小对比
components = ['X', 'obs', 'var', 'raw', 'layers', 'obsm']
sizes = []
labels = []

sizes.append(adata.X.shape[0] * adata.X.shape[1])  # X
labels.append('X\n(cells×genes)')

sizes.append(adata.obs.shape[0] * 10)  # obs 估算
labels.append('obs\n(metdata)')

sizes.append(adata.var.shape[0] * 5)  # var 估算
labels.append('var\n(gene info)')

if adata.raw is not None:
    sizes.append(adata.raw.X.shape[0] * adata.raw.X.shape[1])
else:
    sizes.append(0)
labels.append('raw\n(original)')

sizes.append(1000 * adata.n_obs if adata.layers else 0)  # layers 估算
labels.append('layers\n(variants)')

sizes.append(50 * adata.n_obs if adata.obsm else 0)  # obsm 估算
labels.append('obsm\n(embeddings)')

colors = ['#2ecc71', '#3498db', '#9b59b6', '#f39c12', '#e74c3c', '#1abc9c']
explode = (0.1, 0, 0, 0, 0, 0)  # 突出 X

axes[0].pie(sizes, labels=labels, colors=colors, explode=explode,
            autopct='%1.1f%%', startangle=90, shadow=True)
axes[0].set_title('Data Component Sizes\n(Relative proportions)', fontsize=12)

# 图2: scMAE 数据使用情况
usage_categories = ['Used by scMAE', 'Available but unused', 'Not used']
usage_values = [
    adata.X.shape[0] * adata.X.shape[1],  # X 必需
    (adata.obs.shape[0] * adata.obs.shape[1]) + 
    (adata.var.shape[0] * adata.var.shape[1]) + 
    (adata.raw.X.shape[0] * adata.raw.X.shape[1] if adata.raw else 0),  # 可选
    1000000  # 未使用
]
usage_colors = ['#2ecc71', '#f39c12', '#95a5a6']

bars = axes[1].barh(usage_categories, usage_values, color=usage_colors)
axes[1].set_xlabel('Approximate Data Size (elements)')
axes[1].set_title('scMAE Data Usage Categories', fontsize=12)

# 添加百分比标签
total = sum(usage_values)
for bar, val in zip(bars, usage_values):
    pct = val / total * 100
    axes[1].text(val + total*0.02, bar.get_y() + bar.get_height()/2,
                 f'{pct:.1f}%', va='center')

plt.tight_layout()
plt.show()

---

## 5. 建立自己深度学习模型的注意事项

### 5.1 必需输入数据

In [ ]:
print('=' * 60)
print('🎯 建立深度学习模型 - 必需输入')
print('=' * 60)

print('''
1️⃣  X: 基因表达矩阵
   - 形状: (n_cells, n_genes)
   - 当前数据: ({X.shape[0]}, {X.shape[1]}) - 1000个高度可变基因
   - 数据类型: float32
   - 预处理: Z-score 标准化 (均值≈0, 标准差≈1)

2️⃣  Y: 细胞类型标签 (可选，用于有监督训练或评估)
   - 形状: (n_cells,)
   - 当前类别数: {Y.nunique()}
   - 类别列表: {list(Y.unique())}
''')

# 显示细胞类型编码
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
Y_encoded = le.fit_transform(Y)
print('细胞类型编码对照:')
for i, label in enumerate(le.classes_):
    count = (Y_encoded == i).sum()
    print(f'   {i}: {label} ({count} cells)')

### 5.2 数据预处理流程参考

In [ ]:
print('\n📋 推荐的预处理流程 (参考 scMAE 的 preprocess.py):')
print('=' * 60)
print('''
Step 1: 加载数据
   adata = sc.read_h5ad('data.h5ad')

Step 2: 保存原始数据
   adata.raw = adata.copy()  # 备份原始 counts

Step 3: Per-cell 归一化
   sc.pp.normalize_per_cell(adata)
   # 使每个细胞的 total counts 相同

Step 4: Log1p 变换
   sc.pp.log1p(adata)
   # log(1+x) 稳定方差

Step 5: 高度可变基因筛选
   sc.pp.highly_variable_genes(adata, n_top_genes=1000)
   # 保留生物学变异最大的基因

Step 6: Z-score 标准化
   sc.pp.scale(adata)
   # 每个基因: 均值=0, 标准差=1

Step 7: 提取数据
   X = adata.to_df().values  # (n_cells, n_genes)
   Y = adata.obs['cell_type']  # 细胞类型标签
''')

### 5.3 PyTorch DataLoader 封装示例

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class ScRNADataset(Dataset):
    """单细胞RNA-seq数据集封装 - 参考 scMAE"""
    
    def __init__(self, data, labels=None):
        """
        参数:
            data: numpy array, shape (n_cells, n_genes)
            labels: numpy array, shape (n_cells,), 可选
        """
        self.data = torch.FloatTensor(data)
        self.labels = torch.LongTensor(labels) if labels is not None else None
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        if self.labels is not None:
            return self.data[idx], self.labels[idx]
        return self.data[idx]


# 使用示例
X_np = X.values.astype(np.float32)
Y_np = Y_encoded.astype(np.int64)

dataset = ScRNADataset(X_np, Y_np)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

print(f'数据集大小: {len(dataset)}')
print(f'数据加载器批次数: {len(dataloader)}')

# 测试一个批次
batch_X, batch_Y = next(iter(dataloader))
print(f'\n批次 X 形状: {batch_X.shape}')
print(f'批次 Y 形状: {batch_Y.shape}')
print(f'X 数据范围: [{batch_X.min():.2f}, {batch_X.max():.2f}]')

### 5.4 数据维度要求

In [ ]:
# 当前数据结构
print('=' * 60)
print('📐 当前数据结构')
print('=' * 60)

print(f'''
原始数据:
  - 细胞数: {adata.raw.X.shape[0]:,}
  - 基因数: {adata.raw.X.shape[1]:,}

预处理后 (scMAE 输入):
  - 细胞数: {X.shape[0]:,}
  - 基因数 (HVG): {X.shape[1]:,}

模型层设计参考 (scMAE):
  - 输入层: {X.shape[1]} (基因数)
  - 隐藏层: 128 (可配置)
  - 输出层: {X.shape[1]} (重构原始输入)

如果是自己设计模型:
  - 输入: (batch_size, n_genes) = (任意, {X.shape[1]})
  - 目标输出: (batch_size, n_genes) = (任意, {X.shape[1]})
''')

---

## 6. 数据可视化

In [ ]:
# 预处理后数据的分布
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. 基因表达分布
axes[0, 0].hist(X.values.flatten(), bins=100, color='steelblue', alpha=0.7)
axes[0, 0].set_xlabel('Gene Expression Value')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Gene Expression Values\n(After Z-score normalization)')
axes[0, 0].axvline(x=0, color='red', linestyle='--', label='Mean=0')
axes[0, 0].legend()

# 2. 每个细胞的基因表达量
cell_totals = X.sum(axis=1)
axes[0, 1].hist(cell_totals, bins=50, color='coral', alpha=0.7)
axes[0, 1].set_xlabel('Total Expression per Cell')
axes[0, 1].set_ylabel('Number of Cells')
axes[0, 1].set_title('Total Gene Expression per Cell')

# 3. 每个基因的表达量
gene_totals = X.sum(axis=0)
axes[1, 0].hist(gene_totals, bins=50, color='mediumseagreen', alpha=0.7)
axes[1, 0].set_xlabel('Total Expression per Gene')
axes[1, 0].set_ylabel('Number of Genes')
axes[1, 0].set_title('Total Expression per Gene (HVG)')

# 4. 细胞类型与基因表达关系
cell_type_expr = X.groupby(Y).mean().mean(axis=1)
axes[1, 1].bar(range(len(cell_type_expr)), cell_type_expr.values, color='purple', alpha=0.7)
axes[1, 1].set_xticks(range(len(cell_type_expr)))
axes[1, 1].set_xticklabels(cell_type_expr.index, rotation=45, ha='right')
axes[1, 1].set_ylabel('Mean Gene Expression')
axes[1, 1].set_title('Mean Gene Expression by Cell Type')

plt.tight_layout()
plt.show()

In [ ]:
# PCA 可视化
from sklearn.decomposition import PCA

# PCA 降维
pca = PCA(n_components=50)
X_pca = pca.fit_transform(X.values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. PCA 1 vs PCA 2
scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], 
                          c=Y_encoded, cmap='Set1', alpha=0.6, s=10)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('PCA of Gene Expression\n(Colored by Cell Type)')

# 添加图例
legend_elements = [plt.Line2D([0], [0], marker='o', color='w', 
                              markerfacecolor=plt.cm.Set1(i/len(le.classes_)), 
                              markersize=8, label=label)
                   for i, label in enumerate(le.classes_)]
axes[0].legend(handles=legend_elements, loc='best', fontsize=8)

# 2. 方差解释比例
cumsum = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(1, len(cumsum)+1), cumsum, 'bo-')
axes[1].axhline(y=0.9, color='r', linestyle='--', label='90% variance')
axes[1].set_xlabel('Number of Principal Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('PCA Explained Variance')
axes[1].legend()

plt.tight_layout()
plt.show()

# 打印需要多少 PCs 达到 90% 方差
n_components_90 = np.argmax(cumsum >= 0.9) + 1
print(f'\n达到 90% 方差解释需要 {n_components_90} 个主成分')

In [ ]:
# 热图: 细胞类型 vs 部分基因表达
# 选取方差最大的 20 个基因
gene_vars = X.var()
top_genes = gene_vars.nlargest(20).index.tolist()

# 取每个细胞类型的平均值
X_subset = X[top_genes].T  # genes × cells
cell_type_means = X.groupby(Y).mean()[top_genes]

fig, ax = plt.subplots(figsize=(12, 8))

import seaborn as sns
sns.heatmap(cell_type_means, cmap='RdBu_r', center=0, 
            xticklabels=True, yticklabels=True,
            cbar_kws={'label': 'Mean Expression (Z-score)'})

ax.set_xlabel('Cell Type')
ax.set_ylabel('Top 20 Highly Variable Genes')
ax.set_title('Gene Expression Heatmap by Cell Type\n(Top 20 HVG)')

plt.tight_layout()
plt.show()

---

## 7. 总结

In [ ]:
print('=' * 70)
print('📝 总结')
print('=' * 70)
print(f'''
数据集: SRP182008.h5ad
├── 原始细胞数: {adata.raw.X.shape[0]:,}
├── 原始基因数: {adata.raw.X.shape[1]:,}
├── 细胞类型数: {Y.nunique()}
│
scMAE 算法使用:
├── ✅ 必需: X (预处理后基因表达矩阵, 1000 HVG)
├── ✅ 必需: Y (细胞类型标签)
├── ⚠️ 可选: size_factors (未在模型中使用)
└── ❌ 不使用: raw, obs, var, layers, obsm, varm, obsp, uns

建立自己模型时:
├── 输入: (n_cells, 1000) float32 矩阵
├── 预处理: Z-score 标准化
├── 可选标签: (n_cells,) 细胞类型
└── 可用: adata.raw 原始数据, adata.layers['norm_log'] 备选

模型设计参考:
├── scMAE: Encoder-Decoder 结构, 掩码预测
├── 隐藏层维度: 128 (可调)
└── 损失函数: MSE 重构 + BCE 掩码预测
''')

print('=' * 70)
print('✅ 分析完成!')
print('=' * 70)